In [5]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Planetary Computer / STAC tools
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load

from datetime import date
from tqdm import tqdm
import os
import time
import random

# Setup progress bars
tqdm.pandas()

# Reuse a single STAC client for ESA CCI Land Cover
ESA_CATALOG = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)


def compute_esa_features(row, max_retries: int = 5, base_sleep_s: float = 1.0):
    """Compute ESA CCI Land Cover features for a single sample.

    For the sample's location/date, find the nearest ESA CCI Land Cover item
    and compute summary statistics for each band/variable over a small spatial
    window around the point.
    """
    lat = row["Latitude"]
    lon = row["Longitude"]
    sample_date = pd.to_datetime(row["Sample Date"], dayfirst=True, errors="coerce")

    # Default: all features NaN if anything goes wrong
    default = pd.Series(dtype="float64")

    if pd.isna(sample_date):
        return default

    # ESA CCI is annual; use the sample year
    sample_year = int(sample_date.year)

    # Small buffer (~100m) around the point
    bbox_size = 0.00089831
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2,
    ]

    # Search ESA CCI Land Cover collection
    search = ESA_CATALOG.search(
        collections=["esa-cci-lc"],  # ESA CCI Land Cover on Planetary Computer
        bbox=bbox,
    )
    items = search.item_collection()
    if not items:
        return default

    # Choose item whose year is closest to the sample year
    try:
        items_sorted = sorted(
            items,
            key=lambda x: abs(int(x.properties.get("year", sample_year)) - sample_year),
        )
    except Exception:
        items_sorted = list(items)

    last_err = None
    for attempt in range(max_retries):
        try:
            selected_item = pc.sign(items_sorted[0])

            # Load all bands/variables for this item in the bbox
            data = stac_load([selected_item], bbox=bbox).isel(time=0)

            features = {}
            for var_name, da in data.data_vars.items():
                arr = da.values
                # Flatten and drop NaNs
                flat = arr.ravel()
                if np.issubdtype(flat.dtype, np.floating):
                    flat = flat[np.isfinite(flat)]
                if flat.size == 0:
                    features[f"esa_{var_name}"] = np.nan
                    continue

                # For integer/categorical bands (like land cover class), use mode; else median
                if np.issubdtype(flat.dtype, np.integer):
                    try:
                        mode_val = pd.Series(flat).mode().iloc[0]
                        features[f"esa_{var_name}"] = float(mode_val)
                    except Exception:
                        features[f"esa_{var_name}"] = np.nan
                else:
                    try:
                        features[f"esa_{var_name}"] = float(np.nanmedian(flat))
                    except Exception:
                        features[f"esa_{var_name}"] = np.nan

            return pd.Series(features)

        except Exception as e:
            last_err = e
            sleep_s = base_sleep_s * (2 ** attempt) + random.random() * 0.25
            time.sleep(sleep_s)

    # If all retries fail, return whatever default we have (all NaNs)
    # Uncomment for debugging:
    # print(f"ESA CCI extraction failed after {max_retries} retries: {last_err}")
    return default

In [6]:
# Load training water quality dataset
train_df = pd.read_csv("../water_quality_training_dataset.csv")
print(f"Training dataset shape: {train_df.shape}")
train_df.head()

Training dataset shape: (9319, 6)


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [ ]:
# Chunked ESA CCI extraction for training data (resumable)

chunk_size = 200
output_path_train = "../New Datasets/esa_cci_features_training.csv"

# Helper to count rows in an existing CSV (excluding header)
def count_rows_in_csv(path: str) -> int:
    with open(path, "r", encoding="utf-8") as f:
        return max(sum(1 for _ in f) - 1, 0)

# Determine resume point
start_idx = 0
expected_cols = None
if os.path.exists(output_path_train):
    # Use existing header as the expected column order
    header_cols = pd.read_csv(output_path_train, nrows=0).columns.tolist()
    expected_cols = header_cols
    start_idx = count_rows_in_csv(output_path_train)

print("🚀 Running ESA CCI feature extraction for TRAINING data (chunked)...")
print(f"Total rows in training dataset: {len(train_df)}")
print(f"Output file: {output_path_train}")
print(f"Chunk size: {chunk_size}")
print(f"Resuming from row index: {start_idx}")

for chunk_start in range(start_idx, len(train_df), chunk_size):
    chunk_end = min(chunk_start + chunk_size, len(train_df))
    chunk = train_df.iloc[chunk_start:chunk_end].copy()

    print(f"\nProcessing TRAINING rows {chunk_start}..{chunk_end-1} ({len(chunk)} rows)")

    try:
        # Compute ESA features for this chunk
        esa_feats = chunk.progress_apply(compute_esa_features, axis=1)

        # Attach coordinates and date
        esa_feats["Latitude"] = chunk["Latitude"].values
        esa_feats["Longitude"] = chunk["Longitude"].values
        esa_feats["Sample Date"] = chunk["Sample Date"].values

        # Column ordering: basic columns first, then all ESA features
        base_cols = ["Latitude", "Longitude", "Sample Date"]
        feature_cols = [c for c in esa_feats.columns if c not in base_cols]
        ordered_cols = base_cols + feature_cols

        # If resuming, enforce consistent column order with existing file
        if expected_cols is not None:
            # Add any missing columns as NaN, drop unexpected
            for c in expected_cols:
                if c not in esa_feats.columns:
                    esa_feats[c] = np.nan
            esa_out = esa_feats.reindex(columns=expected_cols)
        else:
            # First chunk and no existing file: define expected_cols
            expected_cols = ordered_cols
            esa_out = esa_feats[expected_cols]

        # Append to CSV (write header only on first write)
        write_header = (not os.path.exists(output_path_train)) or (count_rows_in_csv(output_path_train) == 0)
        esa_out.to_csv(output_path_train, mode="a", header=write_header, index=False)

        # Short pause to be kind to the API
        time.sleep(0.5)

    except Exception as e:
        print(f"\n❌ Chunk failed at TRAINING rows {chunk_start}..{chunk_end-1}: {e}")
        print("You can rerun this cell to resume from the last completed chunk.")
        break

# Preview whatever we have so far
if os.path.exists(output_path_train):
    esa_train_features = pd.read_csv(output_path_train)
    print(f"\n✅ ESA CCI TRAINING rows extracted so far: {len(esa_train_features)}")
    display(esa_train_features.head())
else:
    print("\nNo ESA CCI training output file created yet.")

🚀 Running ESA CCI feature extraction for TRAINING data (chunked)...
Total rows in training dataset: 9319
Output file: ../New Datasets/esa_cci_features_training.csv
Chunk size: 200
Resuming from row index: 3000

Processing TRAINING rows 3000..3199 (200 rows)


100%|██████████| 200/200 [01:36<00:00,  2.07it/s]



Processing TRAINING rows 3200..3399 (200 rows)


100%|██████████| 200/200 [00:52<00:00,  3.80it/s]



Processing TRAINING rows 3400..3599 (200 rows)


...

100%|██████████| 200/200 [01:06<00:00,  2.99it/s]



Processing TRAINING rows 5400..5599 (200 rows)


100%|██████████| 200/200 [00:59<00:00,  3.34it/s]



Processing TRAINING rows 5600..5799 (200 rows)


 50%|████▉     | 99/200 [00:41<04:09,  2.47s/it]

In [ ]:
# Load validation (submission template) dataset
val_df = pd.read_csv("../submission_template.csv")
print(f"Validation dataset shape: {val_df.shape}")
val_df.head()

In [ ]:
# Chunked ESA CCI extraction for validation data (resumable)

chunk_size = 200
output_path_val = "../New Datasets/esa_cci_features_validation.csv"

start_idx = 0
expected_cols_val = None
if os.path.exists(output_path_val):
    header_cols_val = pd.read_csv(output_path_val, nrows=0).columns.tolist()
    expected_cols_val = header_cols_val
    start_idx = count_rows_in_csv(output_path_val)

print("🚀 Running ESA CCI feature extraction for VALIDATION data (chunked)...")
print(f"Total rows in validation dataset: {len(val_df)}")
print(f"Output file: {output_path_val}")
print(f"Chunk size: {chunk_size}")
print(f"Resuming from row index: {start_idx}")

for chunk_start in range(start_idx, len(val_df), chunk_size):
    chunk_end = min(chunk_start + chunk_size, len(val_df))
    chunk = val_df.iloc[chunk_start:chunk_end].copy()

    print(f"\nProcessing VALIDATION rows {chunk_start}..{chunk_end-1} ({len(chunk)} rows)")

    try:
        esa_feats = chunk.progress_apply(compute_esa_features, axis=1)

        esa_feats["Latitude"] = chunk["Latitude"].values
        esa_feats["Longitude"] = chunk["Longitude"].values
        esa_feats["Sample Date"] = chunk["Sample Date"].values

        base_cols = ["Latitude", "Longitude", "Sample Date"]
        feature_cols = [c for c in esa_feats.columns if c not in base_cols]
        ordered_cols = base_cols + feature_cols

        if expected_cols_val is not None:
            for c in expected_cols_val:
                if c not in esa_feats.columns:
                    esa_feats[c] = np.nan
            esa_out = esa_feats.reindex(columns=expected_cols_val)
        else:
            expected_cols_val = ordered_cols
            esa_out = esa_feats[expected_cols_val]

        write_header = (not os.path.exists(output_path_val)) or (count_rows_in_csv(output_path_val) == 0)
        esa_out.to_csv(output_path_val, mode="a", header=write_header, index=False)
        time.sleep(0.5)

    except Exception as e:
        print(f"\n❌ Chunk failed at VALIDATION rows {chunk_start}..{chunk_end-1}: {e}")
        print("You can rerun this cell to resume from the last completed chunk.")
        break

if os.path.exists(output_path_val):
    esa_val_features = pd.read_csv(output_path_val)
    print(f"\n✅ ESA CCI VALIDATION rows extracted so far: {len(esa_val_features)}")
    display(esa_val_features.head())
else:
    print("\nNo ESA CCI validation output file created yet.")